In [ ]:
import os
import torch
from torch_geometric.datasets import Airports, WebKB
from torch_geometric.utils import to_networkx
import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from find_motifs import * 
from Motif_structures_1216 import * 

BASE_DATA_DIR = r'D:\Orbit_degree_LP\jia\SI\Datasets'
os.makedirs(BASE_DATA_DIR, exist_ok=True)

DATASETS_TO_RUN = [
    ('Airports', 'Brazil')
]

moti = []
moti_name = []
for i in range(1, 546):
    func_name = f'M{i}'
    if func_name in globals():
        moti.append(globals()[func_name])
        moti_name.append(func_name)

noti = []
noti_name = []
for i in range(1, 481):
    func_name = f'N{i}'
    if func_name in globals():
        noti.append(globals()[func_name])
        noti_name.append(func_name)

motif_index = list(zip(moti, moti_name))
print(f"初始化完成: 捕获到 {len(moti)} 个边轨道函数, {len(noti)} 个节点轨道函数。")

def process_single_dataset(G, node_labels, dataset_name, output_csv_path):
    num_nodes = G.number_of_nodes()
    nodes_list = list(range(num_nodes))
    adj = G.adj 
    
    X = defaultdict(list)
    X["node_id"] = nodes_list
    X["label"] = node_labels

    print(f"--> [{dataset_name}] 正在计算节点轨道特征 (N系列)...")
    for idx, n_func in enumerate(tqdm(noti, desc=f"{dataset_name} N轨道", leave=False)):
        col_name = noti_name[idx]
        col_values = []
        for node in nodes_list:
            val = node_orbit_motif_degree(G, n_func, node, 1)
            col_values.append(val)
        X[col_name] = col_values

    print(f"--> [{dataset_name}] 正在计算边轨道聚合特征 (M系列)...")
    for idx, m_tuple in enumerate(tqdm(motif_index, desc=f"{dataset_name} M轨道", leave=False)):
        m_func = m_tuple[0]
        m_name = m_tuple[1]
        col_values = []
        
        for u in nodes_list:
            neighbors = list(adj[u])
            if not neighbors:
                col_values.append(0)
                continue
                
            sum_orbit_val = 0
            for v in neighbors:
                val = edge_orbit_motif_degree(G, m_func, (u, v), (1, 2))
                sum_orbit_val += val
            col_values.append(sum_orbit_val)
            
        X[m_name] = col_values

    df_features = pd.DataFrame(X)
    df_features.to_csv(output_csv_path, index=False)
    print(f"[{dataset_name}] 处理完成! 特征形状: {df_features.shape}, 已保存至: {output_csv_path}\n")


for ds_category, ds_name in DATASETS_TO_RUN:
    print(f"========== 开始处理数据集: {ds_category} - {ds_name} ==========")

    ds_path = os.path.join(BASE_DATA_DIR, ds_category)
    if ds_category == 'Airports':
        dataset = Airports(root=ds_path, name=ds_name)
    elif ds_category == 'WebKB':
        dataset = WebKB(root=ds_path, name=ds_name)
    else:
        continue
        
    data = dataset[0]

    node_labels = data.y.numpy().tolist()

    G = to_networkx(data, to_undirected=True)

    output_filename = f"{ds_category.lower()}_{ds_name.lower()}_orbit_features.csv"
    output_csv_path = os.path.join(BASE_DATA_DIR, output_filename)
    
    process_single_dataset(G, node_labels, ds_name, output_csv_path)

In [ ]:
import pandas as pd
import os

INPUT_PATH = r"D:\Orbit_degree_LP\jia\SI\Datasets\airports_brazil_orbit_features.csv"
OUTPUT_PATH = r"D:\Orbit_degree_LP\jia\SI\Datasets\airports_brazil_orbit_features_del.csv"

df = pd.read_csv(INPUT_PATH)

if 'label' not in df.columns:
    raise ValueError("当前文件中没有找到 'label' 列。")

sum_cols = [c for c in df.columns]
print(f"总共有 {len(sum_cols)} 个特征列。")

n_sum_cols = [c for c in sum_cols if c.startswith('N')]
m_sum_cols = [c for c in sum_cols if c.startswith('M')]

print(f"以 'N' 开头的 'sum' 列有 {len(n_sum_cols)} 个，示例：{n_sum_cols[:5]}")
print(f"以 'M' 开头的 'sum' 列有 {len(m_sum_cols)} 个，示例：{m_sum_cols[:5]}")

df_n_sum = df[n_sum_cols].T.drop_duplicates(keep='first').T
n_sum_cols_kept = df_n_sum.columns.tolist()

print(f"去重后保留了 {len(n_sum_cols_kept)} 个以 'N' 开头的列。")

df_m_sum = df[m_sum_cols].T.drop_duplicates(keep='first').T
m_sum_cols_kept = df_m_sum.columns.tolist()

print(f"去重后保留了 {len(m_sum_cols_kept)} 个以 'M' 开头的列。")

df_new = pd.concat([df_n_sum[n_sum_cols_kept], df_m_sum[m_sum_cols_kept], df[['label']]], axis=1)

zero_cols = [c for c in df_new.columns if (df_new[c] == 0).all()]

print(f"检测到 {len(zero_cols)} 个全为 0 的列。")
if zero_cols:
    print("全 0 列示例：", zero_cols[:10])
    df_new = df_new.drop(columns=zero_cols)

cols = df_new.columns.tolist()
cols = ['label'] + [c for c in cols if c != 'label']
df_new = df_new[cols]

df_new.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"去重 + 删除全 0 列后的数据已保存到：{OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
import os

warnings.filterwarnings('ignore')

def select_and_save_is_features(file_path, output_file="airports_brazil_ISM.csv"):
    print(f"正在读取数据: {file_path} ...")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"错误: 无法读取文件. {e}")
        return

    current_basis_cols = ['M1']

    all_m_cols = [f"M{i}" for i in range(1, 546) if f"M{i}" in df.columns]

    def natural_key(string_):
        import re
        return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]
    
    all_m_cols.sort(key=natural_key)
    
    target_cols = [c for c in all_m_cols if c not in current_basis_cols]
    
    print("\n" + "="*60)
    print("开始动态特征筛选 (Dynamic Feature Selection, 使用 Ridge Regression)")
    print(f"初始基底: {current_basis_cols}")
    print(f"待筛选特征数: {len(target_cols)}")
    print("="*60)

    scaler = StandardScaler()
    kept_list = list(current_basis_cols)
    dropped_list = []

    for target in target_cols:
        y = df[target].values
        y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

        basis_data = df[kept_list].values

        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(basis_data)
        X_scaled = scaler.fit_transform(X_poly)

        model = Ridge(alpha=0.01) 
        model.fit(X_scaled, y_scaled)
        y_pred = model.predict(X_scaled)
        r2 = r2_score(y_scaled, y_pred)

        is_redundant = r2 > 0.999  
        
        if is_redundant:
            dropped_list.append(target)
            print(f"[剔除] {target:<5} (R2={r2:.5f}) -> 冗余（可由当前基底近似生成）")
        else:
            kept_list.append(target)
            print(f"[保留] {target:<5} (R2={r2:.5f}) -> 不可约（加入基底，残差非忽略不计）")

    print("\n" + "="*60)
    print("筛选完成")
    print(f"原始特征数: {len(all_m_cols)}")
    print(f"保留特征数: {len(kept_list)}")
    print(f"剔除特征数: {len(dropped_list)}")
    print("-" * 60)
    print(f"最终不可约集合 (ISs): {kept_list}")
    
    out_df = pd.DataFrame(kept_list, columns=['Selected_Features'])
    out_path = os.path.join(os.path.dirname(file_path), output_file)
    out_df.to_csv(out_path, index=False)
    print(f"\n特征列表已保存至: {out_path}")
    
    return kept_list

if __name__ == "__main__":
    file_path = r"D:\Orbit_degree_LP\jia\SI\Datasets\airports_brazil_orbit_features_del.csv"
    final_features = select_and_save_is_features(file_path)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score
import warnings
import os

warnings.filterwarnings('ignore')

def select_and_save_is_features(file_path, output_file="airports_brazil_ISN.csv"):
    print(f"正在读取数据: {file_path} ...")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"错误: 无法读取文件. {e}")
        return

    current_basis_cols = ['N1']

    all_m_cols = [f"N{i}" for i in range(1, 481) if f"N{i}" in df.columns]

    def natural_key(string_):
        import re
        return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]
    
    all_m_cols.sort(key=natural_key)
    
    target_cols = [c for c in all_m_cols if c not in current_basis_cols]
    
    print("\n" + "="*60)
    print("开始动态特征筛选 (Dynamic Feature Selection, 使用 Ridge Regression)")
    print(f"初始基底: {current_basis_cols}")
    print(f"待筛选特征数: {len(target_cols)}")
    print("="*60)

    scaler = StandardScaler()
    kept_list = list(current_basis_cols)
    dropped_list = []

    for target in target_cols:
        y = df[target].values
        y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

        basis_data = df[kept_list].values

        poly = PolynomialFeatures(degree=2, include_bias=False)
        X_poly = poly.fit_transform(basis_data)
        X_scaled = scaler.fit_transform(X_poly)

        model = Ridge(alpha=0.01) 
        model.fit(X_scaled, y_scaled)
        y_pred = model.predict(X_scaled)
        r2 = r2_score(y_scaled, y_pred)

        is_redundant = r2 > 0.999  
        
        if is_redundant:
            dropped_list.append(target)
            print(f"[剔除] {target:<5} (R2={r2:.5f}) -> 冗余（可由当前基底近似生成）")
        else:
            kept_list.append(target)
            print(f"[保留] {target:<5} (R2={r2:.5f}) -> 不可约（加入基底，残差非忽略不计）")

    print("\n" + "="*60)
    print("筛选完成")
    print(f"原始特征数: {len(all_m_cols)}")
    print(f"保留特征数: {len(kept_list)}")
    print(f"剔除特征数: {len(dropped_list)}")
    print("-" * 60)
    print(f"最终不可约集合 (ISs): {kept_list}")
    
    out_df = pd.DataFrame(kept_list, columns=['Selected_Features'])
    out_path = os.path.join(os.path.dirname(file_path), output_file)
    out_df.to_csv(out_path, index=False)
    print(f"\n特征列表已保存至: {out_path}")
    
    return kept_list

if __name__ == "__main__":
    file_path = r"D:\Orbit_degree_LP\jia\SI\Datasets\airports_brazil_orbit_features_del.csv"
    final_features = select_and_save_is_features(file_path)

In [ ]:
from torch_geometric.datasets import WebKB
dataset = WebKB(root='/tmp/Cornell', name='Wisconsin')
graph = dataset[0] 

In [ ]:
import os
import torch
from torch_geometric.datasets import Airports, WebKB
from torch_geometric.utils import to_networkx
import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict

from find_motifs import * 
from Motif_structures_1216 import * 

BASE_DATA_DIR = r'D:\Orbit_degree_LP\jia\SI\Datasets'
os.makedirs(BASE_DATA_DIR, exist_ok=True)

DATASETS_TO_RUN = [
    ('WebKB', 'Wisconsin')
]

print("正在初始化轨道函数列表...")
moti = []
moti_name = []
for i in range(1, 546):
    func_name = f'M{i}'
    if func_name in globals():
        moti.append(globals()[func_name])
        moti_name.append(func_name)

noti = []
noti_name = []
for i in range(1, 481):
    func_name = f'N{i}'
    if func_name in globals():
        noti.append(globals()[func_name])
        noti_name.append(func_name)

motif_index = list(zip(moti, moti_name))
print(f"初始化完成: 捕获到 {len(moti)} 个边轨道函数, {len(noti)} 个节点轨道函数。")

def process_single_dataset(G, node_labels, dataset_name, output_csv_path):
    num_nodes = G.number_of_nodes()
    nodes_list = list(range(num_nodes))
    adj = G.adj 
    
    X = defaultdict(list)
    X["node_id"] = nodes_list
    X["label"] = node_labels

    print(f"--> [{dataset_name}] 正在计算节点轨道特征 (N系列)...")
    for idx, n_func in enumerate(tqdm(noti, desc=f"{dataset_name} N轨道", leave=False)):
        col_name = noti_name[idx]
        col_values = []
        for node in nodes_list:
            val = node_orbit_motif_degree(G, n_func, node, 1)
            col_values.append(val)
        X[col_name] = col_values

    print(f"--> [{dataset_name}] 正在计算边轨道聚合特征 (M系列)...")
    for idx, m_tuple in enumerate(tqdm(motif_index, desc=f"{dataset_name} M轨道", leave=False)):
        m_func = m_tuple[0]
        m_name = m_tuple[1]
        col_values = []
        
        for u in nodes_list:
            neighbors = list(adj[u])
            if not neighbors:
                col_values.append(0)
                continue
                
            sum_orbit_val = 0
            for v in neighbors:
                val = edge_orbit_motif_degree(G, m_func, (u, v), (1, 2))
                sum_orbit_val += val
            col_values.append(sum_orbit_val)
            
        X[m_name] = col_values

    df_features = pd.DataFrame(X)
    df_features.to_csv(output_csv_path, index=False)
    print(f"[{dataset_name}] 处理完成! 特征形状: {df_features.shape}, 已保存至: {output_csv_path}\n")

for ds_category, ds_name in DATASETS_TO_RUN:
    print(f"========== 开始处理数据集: {ds_category} - {ds_name} ==========")

    ds_path = os.path.join(BASE_DATA_DIR, ds_category)
    if ds_category == 'Airports':
        dataset = Airports(root=ds_path, name=ds_name)
    elif ds_category == 'WebKB':
        dataset = WebKB(root=ds_path, name=ds_name)
    else:
        continue
        
    data = dataset[0]

    node_labels = data.y.numpy().tolist()

    G = to_networkx(data, to_undirected=True)

    output_filename = f"{ds_category.lower()}_{ds_name.lower()}_orbit_features.csv"
    output_csv_path = os.path.join(BASE_DATA_DIR, output_filename)
    
    process_single_dataset(G, node_labels, ds_name, output_csv_path)

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

def load_data(dataset_name):
    base_dir = r"D:\Orbit_degree_LP\jia\SI\Datasets"

    orbit_data_path = os.path.join(base_dir, f"{dataset_name}_orbit_features_del.csv")

    isn_data_path = os.path.join(base_dir, f"{dataset_name}_ISN.csv")

    df_orbit = pd.read_csv(orbit_data_path)
    df_isn = pd.read_csv(isn_data_path)

    isn_features = df_isn.iloc[:, 0].dropna().tolist()
    
    return df_orbit, isn_features

def calculate_metrics(df, features, target_col, n_repeats=10):

    empty_res = {k: 0.0 for k in ['acc_mean', 'acc_std', 'prec_mean', 'prec_std', 'rec_mean', 'rec_std', 'f1_mean', 'f1_std']}
    
    if not features or len(features) == 0:
        return empty_res

    valid_features = [f for f in features if f in df.columns]
    if len(valid_features) == 0:
        return empty_res

    X = df[valid_features].values
    y = df[target_col].values
    
    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': []}

    for _ in range(n_repeats):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=np.random.randint(10000))

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=500, 
                                     random_state=42, 
                                     n_jobs=-1)
        clf.fit(X_train_scaled, y_train)

        y_pred = clf.predict(X_test_scaled)
        metrics['acc'].append(accuracy_score(y_test, y_pred))
        metrics['prec'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['rec'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    return {
        'acc_mean': np.mean(metrics['acc']),   'acc_std': np.std(metrics['acc']),
        'prec_mean': np.mean(metrics['prec']), 'prec_std': np.std(metrics['prec']),
        'rec_mean': np.mean(metrics['rec']),   'rec_std': np.std(metrics['rec']),
        'f1_mean': np.mean(metrics['f1']),     'f1_std': np.std(metrics['f1'])
    }

def get_n_index(feature_name):
    match = re.search(r'\d+', str(feature_name))
    if match:
        return int(match.group())
    return -1

def run_experiment_for_dataset(dataset_name, target_col):
    print(f"\n[{dataset_name}] 正在加载和处理数据...")
    try:
        df_orbit, isn_features = load_data(dataset_name)
    except FileNotFoundError as e:
        print(f"找不到数据集文件: {e}")
        return None

    if target_col not in df_orbit.columns:
        print(f"数据集 {dataset_name} 中找不到目标列 '{target_col}'。")
        return None

    isn_order_2 = [f for f in isn_features if get_n_index(f) == 1]
    isn_order_2_3 = [f for f in isn_features if 1 <= get_n_index(f) <= 4]
    isn_order_2_4 = [f for f in isn_features if 1 <= get_n_index(f) <= 15]
    isn_order_2_5 = [f for f in isn_features if 1 <= get_n_index(f) <= 73]
    isn_order_2_6 = [f for f in isn_features if 1 <= get_n_index(f) <= 480]

    raw_all_n = [f for f in df_orbit.columns if str(f).startswith('N') and 1 <= get_n_index(f) <= 480]

    test_groups = [
        ("Selected Order 2 (N1)", isn_order_2),
        ("Selected Order 2-3 (N1-N4)", isn_order_2_3),
        ("Selected Order 2-4 (N1-N15)", isn_order_2_4),
        ("Selected Order 2-5 (N1-N73)", isn_order_2_5),
        ("Selected Order 2-6 (N1-N480)", isn_order_2_6),
        ("Raw Baseline All (N1-N480)", raw_all_n)
    ]

    dataset_results = {}

    print(f"{' ':<33} | {'Dim':<4} | {'Accuracy':<14} | {'Precision':<14} | {'Recall':<14} | {'F1-Score'}")
    print("-" * 110)
    
    for group_name, feats in test_groups:
        res = calculate_metrics(df_orbit, feats, target_col)
        res["dim"] = len(feats)
        dataset_results[group_name] = res
        
        print(f"  --> {group_name:<28} | {res['dim']:<4} | "
              f"{res['acc_mean']:.4f}±{res['acc_std']:.4f} | "
              f"{res['prec_mean']:.4f}±{res['prec_std']:.4f} | "
              f"{res['rec_mean']:.4f}±{res['rec_std']:.4f} | "
              f"{res['f1_mean']:.4f}±{res['f1_std']:.4f}")

    return dataset_results

if __name__ == "__main__":
    datasets = ['airports_brazil', 'airports_europe','airports_usa','webkb_texas', 'webkb_wisconsin'] 
    TARGET_COL = 'label' 
    
    print("="*110)
    print("开始执行节点特征(Node Orbits)跨数据集递增消融实验 (Model: Random Forest, Metrics: All)") 
    print("="*110)

    all_results = {
        "Selected Order 2 (N1)": [],
        "Selected Order 2-3 (N1-N4)": [],
        "Selected Order 2-4 (N1-N15)": [],
        "Selected Order 2-5 (N1-N73)": [],
        "Selected Order 2-6 (N1-N480)": [],
        "Raw Baseline All (N1-N480)": []
    }

    for ds in datasets:
        res = run_experiment_for_dataset(ds, TARGET_COL)
        if res is not None:
            for group_name in all_results.keys():
                all_results[group_name].append(res[group_name])

    print("\n\n" + "="*110)
    print(f"{len(datasets)}个数据集平均性能汇总表 (Feature: Node Orbits, Model: Random Forest)") 
    print("="*110)
    print(f"{'Feature Group':<32} | {'Avg Dim':<7} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
    print("-" * 110)

    for group_name, res_list in all_results.items():
        if len(res_list) == 0:
            continue

        avg_dim = np.mean([x["dim"] for x in res_list])

        acc_vals = [x["acc_mean"] for x in res_list]
        prec_vals = [x["prec_mean"] for x in res_list]
        rec_vals = [x["rec_mean"] for x in res_list]
        f1_vals = [x["f1_mean"] for x in res_list]

        print(f"{group_name:<32} | {avg_dim:<7.1f} | "
              f"{np.mean(acc_vals):.4f}±{np.std(acc_vals):.4f} | "
              f"{np.mean(prec_vals):.4f}±{np.std(prec_vals):.4f} | "
              f"{np.mean(rec_vals):.4f}±{np.std(rec_vals):.4f} | "
              f"{np.mean(f1_vals):.4f}±{np.std(f1_vals):.4f}")
    
    print("="*110 + "\n")

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

def load_data(dataset_name):
    base_dir = r"D:\Orbit_degree_LP\jia\SI\Datasets"
    
    orbit_data_path = os.path.join(base_dir, f"{dataset_name}_orbit_features_del.csv")
    ism_data_path = os.path.join(base_dir, f"{dataset_name}_ISM.csv")
    
    df_orbit = pd.read_csv(orbit_data_path)
    df_ism = pd.read_csv(ism_data_path)
    
    ism_features = df_ism.iloc[:, 0].dropna().tolist()
    
    return df_orbit, ism_features

def calculate_metrics(df, features, target_col, n_repeats=10):
    empty_res = {k: 0.0 for k in ['acc_mean', 'acc_std', 'prec_mean', 'prec_std', 'rec_mean', 'rec_std', 'f1_mean', 'f1_std']}
    
    if not features or len(features) == 0:
        return empty_res

    valid_features = [f for f in features if f in df.columns]
    if len(valid_features) == 0:
        return empty_res

    X = df[valid_features].values
    y = df[target_col].values
    
    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': []}

    for _ in range(n_repeats):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=np.random.randint(10000))
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=500, 
                                     random_state=42, 
                                     n_jobs=-1)
        clf.fit(X_train_scaled, y_train)
        
        y_pred = clf.predict(X_test_scaled)
        metrics['acc'].append(accuracy_score(y_test, y_pred))
        metrics['prec'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['rec'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    return {
        'acc_mean': np.mean(metrics['acc']),   'acc_std': np.std(metrics['acc']),
        'prec_mean': np.mean(metrics['prec']), 'prec_std': np.std(metrics['prec']),
        'rec_mean': np.mean(metrics['rec']),   'rec_std': np.std(metrics['rec']),
        'f1_mean': np.mean(metrics['f1']),     'f1_std': np.std(metrics['f1'])
    }

def get_m_index(feature_name):
    match = re.search(r'\d+', str(feature_name))
    if match:
        return int(match.group())
    return -1

def run_experiment_for_dataset(dataset_name, target_col):
    print(f"\n[{dataset_name}] 正在加载和处理数据...")
    try:
        df_orbit, ism_features = load_data(dataset_name)
    except FileNotFoundError as e:
        print(f"找不到数据集文件: {e}")
        return None

    if target_col not in df_orbit.columns:
        print(f"数据集 {dataset_name} 中找不到目标列 '{target_col}'。")
        return None

    ism_order_3 = [f for f in ism_features if 1 <= get_m_index(f) <= 2]
    ism_order_3_4 = [f for f in ism_features if 1 <= get_m_index(f) <= 12]
    ism_order_3_5 = [f for f in ism_features if 1 <= get_m_index(f) <= 68]
    ism_order_3_6 = [f for f in ism_features if 1 <= get_m_index(f) <= 545]
    
    raw_all_m = [f for f in df_orbit.columns if str(f).startswith('M') and 1 <= get_m_index(f) <= 545]

    test_groups = [
        ("Selected Order 3 (M1-M2)", ism_order_3),
        ("Selected Order 3-4 (M1-M12)", ism_order_3_4),
        ("Selected Order 3-5 (M1-M68)", ism_order_3_5),
        ("Selected Order 3-6 (M1-M545)", ism_order_3_6),
        ("Raw Baseline All (M1-M545)", raw_all_m)
    ]

    dataset_results = {}
    
    print(f"{' ':<33} | {'Dim':<4} | {'Accuracy':<14} | {'Precision':<14} | {'Recall':<14} | {'F1-Score'}")
    print("-" * 110)
    
    for group_name, feats in test_groups:
        res = calculate_metrics(df_orbit, feats, target_col)
        res["dim"] = len(feats)
        dataset_results[group_name] = res
        
        print(f"  --> {group_name:<28} | {res['dim']:<4} | "
              f"{res['acc_mean']:.4f}±{res['acc_std']:.4f} | "
              f"{res['prec_mean']:.4f}±{res['prec_std']:.4f} | "
              f"{res['rec_mean']:.4f}±{res['rec_std']:.4f} | "
              f"{res['f1_mean']:.4f}±{res['f1_std']:.4f}")

    return dataset_results

if __name__ == "__main__":
    datasets = ['airports_brazil', 'airports_europe','airports_usa','webkb_texas', 'webkb_wisconsin'] 
    TARGET_COL = 'label' 
    
    print("="*110)
    print("开始执行边/模体特征(Edge/Motif Orbits)跨数据集递增消融实验 (Model: Random Forest, Metrics: All)") 
    print("="*110)

    all_results = {
        "Selected Order 3 (M1-M2)": [],
        "Selected Order 3-4 (M1-M12)": [],
        "Selected Order 3-5 (M1-M68)": [],
        "Selected Order 3-6 (M1-M545)": [],
        "Raw Baseline All (M1-M545)": []
    }

    for ds in datasets:
        res = run_experiment_for_dataset(ds, TARGET_COL)
        if res is not None:
            for group_name in all_results.keys():
                all_results[group_name].append(res[group_name])

    print("\n\n" + "="*110)
    print(f" {len(datasets)}个数据集平均性能汇总表 (Feature: Edge/Motif Orbits, Model: Random Forest)") 
    print("="*110)
    print(f"{'Feature Group':<32} | {'Avg Dim':<7} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
    print("-" * 110)

    for group_name, res_list in all_results.items():
        if len(res_list) == 0:
            continue
            
        avg_dim = np.mean([x["dim"] for x in res_list])
        
        acc_vals = [x["acc_mean"] for x in res_list]
        prec_vals = [x["prec_mean"] for x in res_list]
        rec_vals = [x["rec_mean"] for x in res_list]
        f1_vals = [x["f1_mean"] for x in res_list]

        print(f"{group_name:<32} | {avg_dim:<7.1f} | "
              f"{np.mean(acc_vals):.4f}±{np.std(acc_vals):.4f} | "
              f"{np.mean(prec_vals):.4f}±{np.std(prec_vals):.4f} | "
              f"{np.mean(rec_vals):.4f}±{np.std(rec_vals):.4f} | "
              f"{np.mean(f1_vals):.4f}±{np.std(f1_vals):.4f}")
    
    print("="*110 + "\n")

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

def load_data(dataset_name):
    base_dir = r"D:\Orbit_degree_LP\jia\SI\Datasets"

    orbit_data_path = os.path.join(base_dir, f"{dataset_name}_orbit_features_del.csv")
    isn_data_path = os.path.join(base_dir, f"{dataset_name}_ISN.csv")
    ism_data_path = os.path.join(base_dir, f"{dataset_name}_ISM.csv")

    if not os.path.exists(orbit_data_path):
        raise FileNotFoundError(f"找不到基础特征文件: {orbit_data_path}")
    df_orbit = pd.read_csv(orbit_data_path)

    isn_features = []
    if os.path.exists(isn_data_path):
        df_isn = pd.read_csv(isn_data_path)
        isn_features = df_isn.iloc[:, 0].dropna().tolist()

    ism_features = []
    if os.path.exists(ism_data_path):
        df_ism = pd.read_csv(ism_data_path)
        ism_features = df_ism.iloc[:, 0].dropna().tolist()
        
    return df_orbit, isn_features, ism_features

def get_index(feature_name):
    match = re.search(r'\d+', str(feature_name))
    if match:
        return int(match.group())
    return -1

def calculate_metrics(df, features, target_col, n_repeats=10):
    empty_res = {k: 0.0 for k in ['acc_mean', 'acc_std', 'prec_mean', 'prec_std', 'rec_mean', 'rec_std', 'f1_mean', 'f1_std']}
    
    if not features or len(features) == 0:
        return empty_res

    valid_features = [f for f in features if f in df.columns]
    if len(valid_features) == 0:
        return empty_res

    X = df[valid_features].values
    y = df[target_col].values
    
    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': []}

    for _ in range(n_repeats):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=np.random.randint(10000))

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=500, 
                                     random_state=42, 
                                     n_jobs=-1)
        clf.fit(X_train_scaled, y_train)
        
        y_pred = clf.predict(X_test_scaled)
        metrics['acc'].append(accuracy_score(y_test, y_pred))
        metrics['prec'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['rec'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    return {
        'acc_mean': np.mean(metrics['acc']),   'acc_std': np.std(metrics['acc']),
        'prec_mean': np.mean(metrics['prec']), 'prec_std': np.std(metrics['prec']),
        'rec_mean': np.mean(metrics['rec']),   'rec_std': np.std(metrics['rec']),
        'f1_mean': np.mean(metrics['f1']),     'f1_std': np.std(metrics['f1'])
    }

def run_experiment_for_dataset(dataset_name, target_col):
    print(f"\n[{dataset_name}] 正在加载和处理数据...")
    try:
        df_orbit, isn_all, ism_all = load_data(dataset_name)
    except FileNotFoundError as e:
        print(f"数据加载失败: {e}")
        return None

    if target_col not in df_orbit.columns:
        print(f"数据集 {dataset_name} 中找不到目标列 '{target_col}'。")
        return None

    isn_filtered = [f for f in isn_all if str(f).startswith('N') and 1 <= get_index(f) <= 480]

    ism_filtered = [f for f in ism_all if str(f).startswith('M') and 1 <= get_index(f) <= 545]

    raw_n = [f for f in df_orbit.columns if str(f).startswith('N') and 1 <= get_index(f) <= 480]
    raw_m = [f for f in df_orbit.columns if str(f).startswith('M') and 1 <= get_index(f) <= 545]
    raw_all = raw_n + raw_m

    test_groups = [
        ("1. Selected ISN + ISM (Fused)", isn_filtered + ism_filtered),
        ("2. Raw Baseline All (N+M)", raw_all)
    ]

    dataset_results = {}

    print(f"{' ':<33} | {'Dim':<4} | {'Accuracy':<14} | {'Precision':<14} | {'Recall':<14} | {'F1-Score'}")
    print("-" * 110)
    
    for group_name, feats in test_groups:
        res = calculate_metrics(df_orbit, feats, target_col)
        res["dim"] = len([f for f in feats if f in df_orbit.columns]) 
        dataset_results[group_name] = res
        
        print(f"  --> {group_name:<28} | {res['dim']:<4} | "
              f"{res['acc_mean']:.4f}±{res['acc_std']:.4f} | "
              f"{res['prec_mean']:.4f}±{res['prec_std']:.4f} | "
              f"{res['rec_mean']:.4f}±{res['rec_std']:.4f} | "
              f"{res['f1_mean']:.4f}±{res['f1_std']:.4f}")

    return dataset_results

if __name__ == "__main__":
    datasets = ['airports_brazil', 'airports_europe','airports_usa','webkb_texas', 'webkb_wisconsin'] 
    TARGET_COL = 'label' 
    
    print("="*110)
    print("开始执行不可约特征联合消融实验 (ISN + ISM Fusion | Model: Random Forest)") 
    print("="*110)

    all_results = {
        "1. Selected ISN + ISM (Fused)": [],
        "2. Raw Baseline All (N+M)": []
    }

    valid_datasets = 0
    for ds in datasets:
        res = run_experiment_for_dataset(ds, TARGET_COL)
        if res is not None:
            valid_datasets += 1
            for group_name in all_results.keys():
                all_results[group_name].append(res[group_name])

    if valid_datasets > 0:
        print("\n\n" + "="*110)
        print(f"{valid_datasets}个数据集平均性能汇总表 (Feature Fusion | Model: Random Forest)") 
        print("="*110)
        print(f"{'Feature Group':<32} | {'Avg Dim':<7} | {'Avg ACC':<14} | {'Avg PREC':<14} | {'Avg REC':<14} | {'Avg F1'}")
        print("-" * 110)

        for group_name, res_list in all_results.items():
            if len(res_list) == 0:
                continue

            avg_dim = np.mean([x["dim"] for x in res_list])

            acc_vals = [x["acc_mean"] for x in res_list]
            prec_vals = [x["prec_mean"] for x in res_list]
            rec_vals = [x["rec_mean"] for x in res_list]
            f1_vals = [x["f1_mean"] for x in res_list]

            print(f"{group_name:<32} | {avg_dim:<7.1f} | "
                  f"{np.mean(acc_vals):.4f}±{np.std(acc_vals):.4f} | "
                  f"{np.mean(prec_vals):.4f}±{np.std(prec_vals):.4f} | "
                  f"{np.mean(rec_vals):.4f}±{np.std(rec_vals):.4f} | "
                  f"{np.mean(f1_vals):.4f}±{np.std(f1_vals):.4f}")
        
        print("="*110 + "\n")
    else:
        print("\n没有成功加载任何数据集，请检查路径和文件名！")